In [1]:
# Set up all required functions
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import dense_rank, desc,when
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,CharType,ArrayType
from itertools import chain

In [2]:
# Initialize system configurations

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]

# Verify BEFORE creating the session

print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.autoBroadcastJoinThreshold", "64MB") \
    .config("spark.local.dir", "D:/spark_tmp") \
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true") \
    .config("spark.python.worker.faulthandler.enabled", "true") \
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")\
    .getOrCreate()



PYSPARK_PYTHON: c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\python.exe
HADOOP_HOME: C:\hadoop


c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
df_read_review_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_review.json")

In [4]:
df_read_tip_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_tip.json")

In [17]:
df_review_data = df_read_review_data.select(F.col("date").alias("review_date"))

In [20]:
df_tip_data = df_read_tip_data.select(F.col("date").alias("tip_date"))

In [34]:
result = df_review_data.join(df_tip_data, on=df_review_data["review_date"] == df_tip_data["tip_date"], how="full") \
    .withColumn("Date", F.coalesce(df_review_data["review_date"], df_tip_data["tip_date"])).drop(F.col("review_date"),F.col("tip_date"))

In [35]:
result.show()

+-------------------+
|               Date|
+-------------------+
|2005-02-16 04:06:26|
|2005-03-01 16:57:17|
|2005-03-01 19:35:44|
|2005-03-09 03:24:06|
|2005-03-09 06:37:47|
|2005-03-09 07:20:08|
|2005-03-10 06:14:37|
|2005-03-15 00:42:22|
|2005-03-16 04:33:03|
|2005-03-22 20:46:09|
|2005-04-10 23:03:48|
|2005-04-11 00:48:09|
|2005-04-13 19:46:35|
|2005-04-15 01:49:42|
|2005-04-21 17:17:48|
|2005-05-04 21:51:38|
|2005-05-11 01:41:26|
|2005-05-11 01:42:19|
|2005-05-11 13:43:08|
|2005-05-11 18:27:40|
+-------------------+
only showing top 20 rows


In [36]:
df_join_tip_review_without_null = result.filter(F.col("Date").isNotNull()).distinct()

In [41]:
df = df_join_tip_review_without_null.withColumn("Year",F.year(F.col("Date")))\
.withColumn("Month",F.month(F.col("Date")))\
.withColumn("Day",F.day(F.col("Date")))

In [38]:
df.show()

+-------------------+----+-----+---+
|               Date|Year|month|day|
+-------------------+----+-----+---+
|2005-02-16 04:06:26|2005|    2| 16|
|2005-03-01 16:57:17|2005|    3|  1|
|2005-03-01 19:35:44|2005|    3|  1|
|2005-03-09 03:24:06|2005|    3|  9|
|2005-03-09 06:37:47|2005|    3|  9|
|2005-03-09 07:20:08|2005|    3|  9|
|2005-03-10 06:14:37|2005|    3| 10|
|2005-03-15 00:42:22|2005|    3| 15|
|2005-03-16 04:33:03|2005|    3| 16|
|2005-03-22 20:46:09|2005|    3| 22|
|2005-04-10 23:03:48|2005|    4| 10|
|2005-04-11 00:48:09|2005|    4| 11|
|2005-04-13 19:46:35|2005|    4| 13|
|2005-04-15 01:49:42|2005|    4| 15|
|2005-04-21 17:17:48|2005|    4| 21|
|2005-05-04 21:51:38|2005|    5|  4|
|2005-05-11 01:41:26|2005|    5| 11|
|2005-05-11 01:42:19|2005|    5| 11|
|2005-05-11 13:43:08|2005|    5| 11|
|2005-05-11 18:27:40|2005|    5| 11|
+-------------------+----+-----+---+
only showing top 20 rows


In [42]:
df_add_surrogate_key = df.withColumn("Date_pk_sk", F.monotonically_increasing_id() + 1)

In [40]:
df_add_surrogate_key.show()

+-------------------+----+-----+---+----------+
|               Date|Year|month|day|Date_pk_sk|
+-------------------+----+-----+---+----------+
|2005-02-16 04:06:26|2005|    2| 16|         1|
|2005-03-01 16:57:17|2005|    3|  1|         2|
|2005-03-01 19:35:44|2005|    3|  1|         3|
|2005-03-09 03:24:06|2005|    3|  9|         4|
|2005-03-09 06:37:47|2005|    3|  9|         5|
|2005-03-09 07:20:08|2005|    3|  9|         6|
|2005-03-10 06:14:37|2005|    3| 10|         7|
|2005-03-15 00:42:22|2005|    3| 15|         8|
|2005-03-16 04:33:03|2005|    3| 16|         9|
|2005-03-22 20:46:09|2005|    3| 22|        10|
|2005-04-10 23:03:48|2005|    4| 10|        11|
|2005-04-11 00:48:09|2005|    4| 11|        12|
|2005-04-13 19:46:35|2005|    4| 13|        13|
|2005-04-15 01:49:42|2005|    4| 15|        14|
|2005-04-21 17:17:48|2005|    4| 21|        15|
|2005-05-04 21:51:38|2005|    5|  4|        16|
|2005-05-11 01:41:26|2005|    5| 11|        17|
|2005-05-11 01:42:19|2005|    5| 11|    

In [43]:
df_final = df_add_surrogate_key.select(F.col("Date_pk_sk") , F.col("Date") , F.col("Year") , F.col("Month") , F.col("Day") )

In [44]:
df_final.write.mode("overwrite").parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_date")

In [ ]:
df_final.show()